In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

profile = pd.read_csv('../../results/series_profile.csv')
colors = {'A': '#2ecc71', 'B': '#f39c12', 'C': '#e74c3c', 'D': '#95a5a6'}
print(f'Total combinations: {len(profile)}')
profile.head()

## 1. Class Distribution

In [ ]:
class_counts = profile['series_class'].value_counts().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(class_counts.index, class_counts.values, color=[colors[c] for c in class_counts.index])
for i, v in enumerate(class_counts.values):
    ax1.text(i, v + 50, str(v), ha='center', fontweight='bold')
ax1.set_title('Series Count per Class')

ax2.pie(class_counts.values, labels=[f'{c} ({v})' for c, v in zip(class_counts.index, class_counts.values)],
        colors=[colors[c] for c in class_counts.index], autopct='%1.1f%%')
ax2.set_title('Distribution')

plt.tight_layout()
plt.show()

print('A = Strong (6+ months, fill >80%, active)')
print('B = Moderate (3+ months, fill >50%)')
print('C = Weak (short history or sparse)')
print('D = Dead (no sales last 30 days)')

## 2. Summary per Class

In [ ]:
profile.groupby('series_class').agg(
    count=('series_class', 'size'),
    avg_history_days=('history_days', 'mean'),
    avg_fill_rate=('fill_rate', 'mean'),
    avg_fill_rate_last_30=('fill_rate_last_30', 'mean'),
    avg_mean_sales=('mean_sales', 'mean'),
    avg_days_since_last_sale=('days_since_last_sale', 'mean'),
).round(2)

## 3. Fill Rate

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col, title in zip(axes,
    ['fill_rate', 'fill_rate_last_30', 'fill_rate_last_90'],
    ['Full History', 'Last 30 Days', 'Last 90 Days']):
    for cls in sorted(profile['series_class'].unique()):
        data = profile[profile['series_class'] == cls][col]
        ax.hist(data, bins=30, alpha=0.5, label=f'{cls}', color=colors[cls])
    ax.set_title(f'Fill Rate - {title}')
    ax.legend()

plt.tight_layout()
plt.show()

## 4. History Length

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for cls in sorted(profile['series_class'].unique()):
    data = profile[profile['series_class'] == cls]['history_days']
    ax.hist(data, bins=30, alpha=0.5, label=f'{cls}', color=colors[cls])
ax.set_title('History Length (days)')
ax.set_xlabel('Days')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Days Since Last Sale

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for cls in sorted(profile['series_class'].unique()):
    data = profile[profile['series_class'] == cls]['days_since_last_sale'].clip(upper=200)
    ax.hist(data, bins=40, alpha=0.5, label=f'{cls}', color=colors[cls])
ax.set_title('Days Since Last Sale')
ax.set_xlabel('Days')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Per-Product: stores by class

In [ ]:
product_class = profile.groupby('codigo_barras_sku')['series_class'].value_counts().unstack(fill_value=0)
product_class['total'] = product_class.sum(axis=1)
product_class = product_class.sort_values('total', ascending=False).head(20)

cols_to_plot = [c for c in ['A', 'B', 'C', 'D'] if c in product_class.columns]
product_class[cols_to_plot].plot(
    kind='barh', stacked=True, figsize=(12, 6),
    color=[colors[c] for c in cols_to_plot]
)
plt.title('Top 20 Products: Stores by Class')
plt.xlabel('Number of Stores')
plt.legend(title='Class')
plt.tight_layout()
plt.show()

## 7. Trainability Summary

In [ ]:
for cls in ['A', 'B', 'C', 'D']:
    n = (profile['series_class'] == cls).sum()
    pct = n / len(profile) * 100
    print(f'Class {cls}: {n:>6} ({pct:.1f}%)')

trainable = profile[profile['series_class'].isin(['A', 'B'])]
print(f'\nTrainable (A+B): {len(trainable)} ({len(trainable)/len(profile)*100:.1f}%)')
print(f'  Products: {trainable["codigo_barras_sku"].nunique()}')
print(f'  Stores:   {trainable["pdv_anonimizado"].nunique()}')